# Programming-Language Samples Demo

Browse 300 cached code samples across Python, JavaScript, Java, C++, Go, and Rust: 50 per language, each with 2,000 Llama-3.2-3B content tokens.

Source: [StarCoderData](https://huggingface.co/datasets/bigcode/starcoderdata), training split of the `python`, `javascript`, `java`, `cpp`, `go`, and `rust` subsets. Each sample is a contiguous window from one source file; languages are sampled independently.

Light cleanup normalized line endings and removed byte-order marks and stray control characters, preserving indentation, spacing, and comments.

Run all cells from `Programming-language-inputs` to browse cached code and source details. Diagnostics are computed by `Generate-programming-language-spectral-diagnostics.ipynb`.

In [1]:
from pathlib import Path
from typing import Any, Dict, Optional

import html as _html
import json

from IPython.display import HTML, clear_output, display

PREVIEW_TOKENS = 500
DATA_DIR = Path("programming_language_samples")
COMBINED_JSONL_PATH = DATA_DIR / "programming_language_2000_token_windows.jsonl"
LANGUAGE_NAMES = {
    "python": "Python",
    "javascript": "JavaScript",
    "java": "Java",
    "cpp": "C++",
    "go": "Go",
    "rust": "Rust"
}
LANGUAGES = list(LANGUAGE_NAMES)

if not COMBINED_JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Cached sample file not found: {COMBINED_JSONL_PATH.name}. "
        "Run this notebook from Programming-language-inputs, or set DATA_DIR to the sample folder. "
        "If the file is missing, restore the combined JSONL distributed with this demo."
    )

with COMBINED_JSONL_PATH.open(encoding="utf-8") as sample_file:
    browser_records = [json.loads(line) for line in sample_file if line.strip()]
if not browser_records:
    raise ValueError(f"The cached sample file is empty: {COMBINED_JSONL_PATH}")

browser_records_by_run_language = {}
required_fields = {"sample_id", "run_id", "language", "text", "token_ids", "num_tokens", "preview_text_500_tokens"}
for record in browser_records:
    missing_fields = required_fields.difference(record)
    if missing_fields:
        raise ValueError(
            f"A cached sample is missing fields: {sorted(missing_fields)}. "
            "Restore the combined JSONL distributed with this demo."
        )
    if not isinstance(record["text"], str) or not isinstance(record["preview_text_500_tokens"], str):
        raise ValueError(f"Cached display text must be a string for {record['sample_id']}.")
    if len(record["token_ids"]) != record["num_tokens"]:
        raise ValueError(f"Saved token counts disagree for {record['sample_id']}.")
    key = (int(record["run_id"]), str(record["language"]))
    if key in browser_records_by_run_language:
        raise ValueError(f"Duplicate sample for run and language: {key}")
    if key[1] not in LANGUAGE_NAMES:
        raise ValueError(f"Unrecognized cached language: {key[1]}")
    browser_records_by_run_language[key] = record

run_options = sorted({key[0] for key in browser_records_by_run_language})
language_options = [language for language in LANGUAGES if any(key[1] == language for key in browser_records_by_run_language)]
if language_options != LANGUAGES or any(
    (run_id, language) not in browser_records_by_run_language
    for run_id in run_options for language in language_options
):
    raise ValueError("Each cached sample must be available in all six languages.")

print(f"Loaded {len(browser_records):,} cached samples: {len(run_options)} per language.")

Loaded 300 cached samples: 50 per language.


Select a language and sample to view the first 500 tokens or the full 2,000-token code, with its repository and file path. Both views load directly from cached samples. Interactive controls require `ipywidgets`; static previews below also work without widgets.

In [2]:
def preview_text(record: Dict[str, Any], preview_tokens: Optional[int] = PREVIEW_TOKENS) -> str:
    if preview_tokens is None:
        text = record["text"]
    elif preview_tokens == PREVIEW_TOKENS:
        text = record["preview_text_500_tokens"]
    else:
        raise ValueError("Choose the cached first 500 tokens or the full sample.")
    return text


def render_sample(run_id: int, language: str, preview_tokens: Optional[int]) -> HTML:
    record = browser_records_by_run_language[(int(run_id), str(language))]
    title = (
        f"{language} - {LANGUAGE_NAMES.get(language, language)} | "
        f"run {int(run_id):02d} | "
        f"{_html.escape(str(record.get('repository_name') or '<repo unavailable>'))} | "
        f"{_html.escape(str(record.get('source_file_path') or '<path unavailable>'))}"
    )
    # Escape source code for HTML display and preserve its spacing in a preformatted block.
    text = _html.escape(preview_text(record, preview_tokens))
    style = """
    <style>
      .plsd-wrap {
        border: 1px solid #d5d7dc;
        border-radius: 8px;
        overflow: hidden;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
        background: #ffffff;
        width: 100%;
      }
      .plsd-head {
        background: #f5f6f8;
        border-bottom: 1px solid #d5d7dc;
        font-weight: 650;
        color: #1f2937;
        padding: 10px 12px;
      }
      .plsd-code {
        margin: 0;
        padding: 12px 14px;
        white-space: pre-wrap;
        overflow-wrap: normal;
        tab-size: 4;
        line-height: 1.45;
        color: #111827;
        max-height: 640px;
        overflow: auto;
        font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', monospace;
        font-size: 13px;
      }
    </style>
    """
    return HTML(
        style
        + "<div class='plsd-wrap'>"
        + f"<div class='plsd-head'>{title}</div>"
        + f"<pre class='plsd-code'>{text}</pre>"
        + "</div>"
    )


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("Interactive controls are unavailable. The static previews below still work.")

if widgets is not None:
    language_dropdown = widgets.Dropdown(
        options=[(f"{language} - {LANGUAGE_NAMES.get(language, language)}", language) for language in language_options],
        value=language_options[0],
        description="Language",
        layout=widgets.Layout(width="260px"),
    )
    sample_dropdown = widgets.Dropdown(
        options=[(f"Sample {run_id:02d}", int(run_id)) for run_id in run_options],
        value=int(run_options[0]),
        description="Run",
        layout=widgets.Layout(width="180px"),
    )
    view_dropdown = widgets.Dropdown(
        options=[("First 500", PREVIEW_TOKENS), ("Full", None)],
        value=PREVIEW_TOKENS,
        description="View",
        layout=widgets.Layout(width="180px"),
    )
    output = widgets.Output(layout=widgets.Layout(width="100%"))


    def render_selected_sample(*_):
        with output:
            clear_output(wait=True)
            display(render_sample(sample_dropdown.value, language_dropdown.value, view_dropdown.value))


    language_dropdown.observe(render_selected_sample, names="value")
    sample_dropdown.observe(render_selected_sample, names="value")
    view_dropdown.observe(render_selected_sample, names="value")

    render_selected_sample()
    sample_browser = widgets.VBox([
        widgets.HBox([language_dropdown, sample_dropdown, view_dropdown]),
        output,
    ])
    display(sample_browser)

Static previews show one sample per programming language with its original formatting. Change `PREVIEW_RUN_ID` to choose another sample or set `STATIC_PREVIEW_TOKENS = None` to show full code.

In [3]:
PREVIEW_RUN_ID = run_options[0]
STATIC_PREVIEW_TOKENS = PREVIEW_TOKENS

for language in language_options:
    display(render_sample(PREVIEW_RUN_ID, language, STATIC_PREVIEW_TOKENS))